# LightGBM — Prédiction de la consommation électrique
**Sous-ensemble ciblé :** Maisons individuelles plein pied, chauffage électrique, sans VE / piscine / PV  
**Cible :** `out.electricity.total.energy_consumption..kwh`  
**Stratification :** Zones climatiques ASHRAE IECC 2004 (17 zones)

## Étape 1 — Chargement & filtrage

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import lightgbm as lgb

In [ ]:
ROOT           = Path().resolve().parent.parent
DATA_PROCESSED = ROOT / 'data' / 'processed'

X    = pd.read_parquet(DATA_PROCESSED / 'X.parquet')
Y    = pd.read_parquet(DATA_PROCESSED / 'Y.parquet')
meta = pd.read_parquet(DATA_PROCESSED / 'metadata_clean.parquet')

print(f'X : {X.shape} | Y : {Y.shape} | meta : {meta.shape}')

In [ ]:
# Masque : maisons individuelles plein pied, chauffage électrique, sans VE / piscine / PV
mask = (
    (meta['in.geometry_building_type_recs'] == 'Single-Family Detached') &
    (X['in.geometry_stories'] == 1)                                      &
    (meta['in.heating_fuel']  == 'Electricity')                          &
    (X['in.electric_vehicle_charger'] == 0)                              &
    (X['in.has_pool'] == 0)                                              &
    (X['in.has_pv']   == 0)
)

X_sub    = X[mask].reset_index(drop=True)
Y_sub    = Y[mask].reset_index(drop=True)
meta_sub = meta[mask].reset_index(drop=True)

# Convertir les colonnes PyArrow en types numpy standards (compatibilité sklearn)
def to_numpy_dtypes(df):
    for col in df.columns:
        dt = df[col].dtype
        if hasattr(dt, 'numpy_dtype'):
            df[col] = df[col].astype(dt.numpy_dtype)
        elif hasattr(dt, 'pyarrow_dtype'):
            df[col] = df[col].astype(str(dt.pyarrow_dtype))
    return df

X_sub = to_numpy_dtypes(X_sub.copy())
Y_sub = to_numpy_dtypes(Y_sub.copy())

print(f'Sous-ensemble : {len(X_sub):,} logements / {len(X):,} total ({len(X_sub)/len(X)*100:.1f}%)')

## Étape 2 — Split train / val / test stratifié par zone ASHRAE

In [ ]:
TARGET = 'out.electricity.total.energy_consumption..kwh'

# .astype(str).to_numpy() pour éviter le format PyArrow qui bloque sklearn
ashrae = meta_sub['in.ashrae_iecc_climate_zone_2004'].astype(str).to_numpy()

print('Distribution ASHRAE dans le sous-ensemble :')
print(pd.Series(ashrae).value_counts().sort_index().to_string())
print(f'\n{pd.Series(ashrae).nunique()} zones représentées')

In [ ]:
# Split train+val / test (80 / 20) — stratifié par zone ASHRAE
X_trainval, X_test, Y_trainval, Y_test, ashrae_trainval, _ = train_test_split(
    X_sub,
    Y_sub[TARGET],
    ashrae,
    test_size=0.2,
    random_state=42,
    stratify=ashrae
)

# Split train / val (80 / 20 du trainval) — stratifié par zone ASHRAE
X_train, X_val, Y_train, Y_val = train_test_split(
    X_trainval,
    Y_trainval,
    test_size=0.2,
    random_state=42,
    stratify=ashrae_trainval
)

total = len(X_train) + len(X_val) + len(X_test)
print(f'Train : {X_train.shape}  ({len(X_train)/total:.1%})')
print(f'Val   : {X_val.shape}  ({len(X_val)/total:.1%})')
print(f'Test  : {X_test.shape}  ({len(X_test)/total:.1%})')
print(f'\nY_train — mean: {Y_train.mean():.0f} kWh | std: {Y_train.std():.0f} kWh')

## Étape 2 — Analyse de la cible
### Sous-étape 1 — Distribution de Y (électricité)

In [ ]:
# Stats descriptives
stats = Y_train.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9])
print('=== Statistiques descriptives — Y_train (kWh/an) ===')
print(stats.round(1).to_string())
print(f'\nSkewness  : {Y_train.skew():.3f}')
print(f'Kurtosis  : {Y_train.kurtosis():.3f}')

# Histogramme
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(Y_train, bins=80, color='steelblue', edgecolor='white', linewidth=0.3)
axes[0].axvline(Y_train.mean(),   color='red',    linestyle='--', linewidth=1.5, label=f'Moyenne  {Y_train.mean():.0f} kWh')
axes[0].axvline(Y_train.median(), color='orange', linestyle='--', linewidth=1.5, label=f'Médiane  {Y_train.median():.0f} kWh')
axes[0].set_title('Distribution brute de Y (électricité)')
axes[0].set_xlabel('Consommation (kWh/an)')
axes[0].set_ylabel('Nombre de logements')
axes[0].legend()

axes[1].hist(Y_train, bins=80, color='steelblue', edgecolor='white', linewidth=0.3)
axes[1].set_yscale('log')
axes[1].set_title('Distribution brute — échelle log (axe Y)')
axes[1].set_xlabel('Consommation (kWh/an)')
axes[1].set_ylabel('Nombre de logements (log)')

plt.tight_layout()
plt.show()

### Sous-étape 2 — Transformation log

In [ ]:
Y_train_log = np.log1p(Y_train)
Y_val_log   = np.log1p(Y_val)
Y_test_log  = np.log1p(Y_test)

# Comparaison des skewness
print('=== Comparaison Brut vs Log ===')
print(f'Skewness brut  : {Y_train.skew():.3f}')
print(f'Skewness log   : {Y_train_log.skew():.3f}')
print(f'Kurtosis brut  : {Y_train.kurtosis():.3f}')
print(f'Kurtosis log   : {Y_train_log.kurtosis():.3f}')

# Visualisation côte à côte
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(Y_train,     bins=80, color='steelblue', edgecolor='white', linewidth=0.3)
axes[0].set_title(f'Y brut  — skewness = {Y_train.skew():.2f}')
axes[0].set_xlabel('Consommation (kWh/an)')
axes[0].set_ylabel('Nombre de logements')

axes[1].hist(Y_train_log, bins=80, color='seagreen',  edgecolor='white', linewidth=0.3)
axes[1].set_title(f'log(Y+1) — skewness = {Y_train_log.skew():.2f}')
axes[1].set_xlabel('log(Consommation + 1)')
axes[1].set_ylabel('Nombre de logements')

plt.tight_layout()
plt.show()

### Sous-étape 3 — Distribution par zone ASHRAE

In [ ]:
# Associer la zone ASHRAE au Y_train via les index
ashrae_train_labels = meta_sub.iloc[X_train.index]['in.ashrae_iecc_climate_zone_2004'].astype(str).values

df_ashrae = pd.DataFrame({
    'zone'  : ashrae_train_labels,
    'conso' : Y_train.values
})

# Médiane par zone pour trier
ordre = df_ashrae.groupby('zone')['conso'].median().sort_values().index

# Stats par zone
print('=== Médiane kWh/an par zone ASHRAE ===')
print(df_ashrae.groupby('zone')['conso'].agg(['median', 'mean', 'std', 'count']).round(0).loc[ordre].to_string())

# Boxplot
fig, ax = plt.subplots(figsize=(14, 5))
data_par_zone = [df_ashrae[df_ashrae['zone'] == z]['conso'].values for z in ordre]

bp = ax.boxplot(data_par_zone, labels=ordre, patch_artist=True,
                flierprops=dict(marker='.', markersize=2, alpha=0.3))

for patch in bp['boxes']:
    patch.set_facecolor('steelblue')
    patch.set_alpha(0.6)

ax.set_title('Distribution de la consommation électrique par zone ASHRAE\n(maisons individuelles plein pied, chauffage électrique)')
ax.set_xlabel('Zone ASHRAE IECC 2004')
ax.set_ylabel('Consommation (kWh/an)')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### Sous-étape 4 — Corrélation features vs Y

In [ ]:
# Corrélation de Pearson entre chaque feature numérique et Y_train
X_train_num = X_train.select_dtypes(include='number')

corr = X_train_num.corrwith(Y_train).dropna().sort_values(key=abs, ascending=False)

print(f'=== Top 15 features corrélées avec la conso électrique ===\n')
print(corr.head(15).round(3).to_string())
print(f'\n=== Top 5 corrélations négatives ===\n')
print(corr[corr < 0].head(5).round(3).to_string())

# Barplot
fig, ax = plt.subplots(figsize=(10, 6))

top15 = corr.head(15)
colors = ['tomato' if v < 0 else 'steelblue' for v in top15.values]
ax.barh(top15.index[::-1], top15.values[::-1], color=colors[::-1])
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Top 15 features — corrélation avec la consommation électrique')
ax.set_xlabel('Corrélation de Pearson')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## Étape 3 — Baseline LightGBM

In [ ]:
# ── Définition du modèle ──────────────────────────────────────────────────────
# LGBMRegressor = LightGBM pour la régression (prédire un nombre, pas une classe)
# Ces hyperparamètres sont les valeurs par défaut — on ne les a pas encore optimisés
model = lgb.LGBMRegressor(
    n_estimators  = 1000,   # nombre maximum d'arbres à construire
    learning_rate = 0.05,   # pas d'apprentissage : plus petit = plus précis mais plus lent
    num_leaves    = 63,     # complexité de chaque arbre : plus élevé = arbre plus profond
    random_state  = 42,     # reproductibilité des résultats
    n_jobs        = -1      # utilise tous les cœurs CPU disponibles
)

# ── Entraînement ──────────────────────────────────────────────────────────────
# eval_set : on donne la val au modèle pour surveiller ses performances pendant l'entraînement
# early_stopping(50) : si la val ne s'améliore pas pendant 50 arbres consécutifs → on arrête
# Cela évite l'overfitting (le modèle qui apprend par cœur le train sans généraliser)
model.fit(
    X_train, Y_train,
    eval_set=[(X_val, Y_val)],
    callbacks=[lgb.early_stopping(50, verbose=False),
               lgb.log_evaluation(100)]   # affiche le score toutes les 100 itérations
)

print(f'\nNombre d\'arbres utilisés : {model.best_iteration_} / {model.n_estimators}')

In [ ]:
# ── Prédictions ───────────────────────────────────────────────────────────────
# On prédit sur les 3 ensembles pour comparer train vs val vs test
pred_train = model.predict(X_train)
pred_val   = model.predict(X_val)
pred_test  = model.predict(X_test)

# ── Métriques ─────────────────────────────────────────────────────────────────
# R²   : 1.0 = parfait | 0.0 = aussi mauvais que prédire la moyenne | <0 = pire que la moyenne
# RMSE : erreur quadratique moyenne (en kWh) — pénalise les grandes erreurs
# MAE  : erreur absolue moyenne (en kWh) — plus robuste aux outliers

def metriques(y_true, y_pred, label):
    r2   = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    print(f'{label:10} | R² = {r2:.4f} | RMSE = {rmse:8.0f} kWh | MAE = {mae:7.0f} kWh')
    return r2, rmse, mae

print('=== Baseline LightGBM — Résultats ===\n')
r2_train, rmse_train, mae_train = metriques(Y_train, pred_train, 'Train')
r2_val,   rmse_val,   mae_val   = metriques(Y_val,   pred_val,   'Val')
r2_test,  rmse_test,  mae_test  = metriques(Y_test,  pred_test,  'Test')

print(f'\nOverfit gap (Train R² - Test R²) : {r2_train - r2_test:.4f}')

## Étape 5 — Optimisation des hyperparamètres avec Optuna

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)  # silence les logs verbose

def objective(trial):
    """
    Optuna appelle cette fonction à chaque essai.
    Elle propose des hyperparamètres, entraîne un modèle et retourne le RMSE val.
    Optuna cherche à MINIMISER cette valeur.
    """
    params = {
        # Nombre d'arbres — on laisse l'early stopping décider quand s'arrêter
        'n_estimators'     : 3000,

        # Taux d'apprentissage — plus petit = plus précis mais plus lent
        'learning_rate'    : trial.suggest_float('learning_rate', 0.01, 0.1, log=True),

        # Complexité de chaque arbre — contrôle la capacité du modèle
        'num_leaves'       : trial.suggest_int('num_leaves', 20, 200),

        # Nombre minimum de logements dans une feuille — régularisation contre l'overfitting
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 200),

        # Proportion de features utilisées par arbre — réduit la corrélation entre arbres
        'colsample_bytree' : trial.suggest_float('colsample_bytree', 0.5, 1.0),

        # Proportion de logements utilisés par arbre — évite de mémoriser des cas rares
        'subsample'        : trial.suggest_float('subsample', 0.5, 1.0),

        # Régularisation L2 — pénalise les poids trop grands
        'reg_lambda'       : trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),

        # Régularisation L1 — force certains poids à zéro (sélection de features)
        'reg_alpha'        : trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),

        'random_state': 42,
        'n_jobs'      : -1,
        'verbose'     : -1,
    }

    m = lgb.LGBMRegressor(**params)
    m.fit(
        X_train, Y_train,
        eval_set=[(X_val, Y_val)],
        callbacks=[lgb.early_stopping(50, verbose=False)]
    )

    pred_val = m.predict(X_val)
    return np.sqrt(mean_squared_error(Y_val, pred_val))   # RMSE val à minimiser


# Lancer l'optimisation — 50 essais (~5-10 min)
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f'\n=== Meilleurs hyperparamètres trouvés ===')
print(f'RMSE val : {study.best_value:.0f} kWh')
for k, v in study.best_params.items():
    print(f'  {k:25} : {v}')

### Réentraînement avec les meilleurs hyperparamètres

In [ ]:
# Réentraînement sur train+val avec les meilleurs paramètres trouvés par Optuna
# On utilise X_trainval (train + val réunis) pour maximiser les données d'entraînement
# Le nombre d'arbres optimal est celui trouvé par l'early stopping pendant Optuna
best_params = study.best_params
best_params['n_estimators'] = 3000
best_params['random_state'] = 42
best_params['n_jobs']       = -1
best_params['verbose']      = -1

model_opt = lgb.LGBMRegressor(**best_params)

model_opt.fit(
    X_trainval, Y_trainval,
    eval_set=[(X_test, Y_test)],
    callbacks=[lgb.early_stopping(50, verbose=False),
               lgb.log_evaluation(200)]
)

# ── Évaluation finale ─────────────────────────────────────────────────────────
pred_opt_train = model_opt.predict(X_trainval)
pred_opt_test  = model_opt.predict(X_test)

print('=== Modèle optimisé — Résultats finaux ===\n')
r2_opt_train, rmse_opt_train, _ = metriques(Y_trainval, pred_opt_train, 'Train+Val')
r2_opt_test,  rmse_opt_test,  _ = metriques(Y_test,     pred_opt_test,  'Test')
print(f'\nOverfit gap : {r2_opt_train - r2_opt_test:.4f}')

print('\n=== Comparaison Baseline vs Optimisé (Test) ===\n')
print(f'{"":15} {"Baseline":>12} {"Optimisé":>12} {"Gain":>10}')
print(f'{"R²":15} {r2_test:>12.4f} {r2_opt_test:>12.4f} {r2_opt_test - r2_test:>+10.4f}')
print(f'{"RMSE (kWh)":15} {rmse_test:>12.0f} {rmse_opt_test:>12.0f} {rmse_opt_test - rmse_test:>+10.0f}')

## Étape 6 — Modèle convergé (early stopping sur n_estimators illimité)

**Problème identifié :** le modèle optimisé utilisait encore tous les 3 000 arbres → early stopping n'a jamais déclenché → le modèle n'avait pas convergé.

**Solution :** augmenter `n_estimators` à 10 000 et la patience à 100. L'early stopping s'arrêtera automatiquement quand la val ne s'améliorera plus.

## Étape 7 — Réduction de l'overfitting : Optuna + K-Fold CV

**Diagnostic :** Train+Val R²=0.985 vs Test R²=0.926 → gap de 0.059 → le modèle mémorise le train.

**Deux causes :**
1. Optuna était calibré sur **un seul split val** → hyperparamètres trop ajustés à cette partition
2. Les bornes de régularisation étaient trop permissives (reg_lambda jusqu'à 10 seulement)

**Solution :**
- **K-Fold CV (5 folds)** dans Optuna → les hyperparamètres sont évalués sur 5 splits différents → plus robustes
- Bornes de régularisation plus agressives : `min_child_samples` ↑, `reg_lambda` ↑, `num_leaves` ↓

In [ ]:
# Reprendre les meilleurs hyperparamètres Optuna et augmenter n_estimators à 10 000
# Early stopping avec patience=100 : s'arrête dès que la val stagne 100 arbres
params_convergé = dict(study.best_params)
params_convergé.update({
    'n_estimators' : 10_000,   # plafond élevé — early stopping décidera le bon nombre
    'random_state' : 42,
    'n_jobs'       : -1,
    'verbose'      : -1,
})

model_conv = lgb.LGBMRegressor(**params_convergé)

model_conv.fit(
    X_trainval, Y_trainval,
    eval_set   = [(X_test, Y_test)],
    callbacks  = [
        lgb.early_stopping(100, verbose=False),   # patience doublée
        lgb.log_evaluation(500),                  # log toutes les 500 itérations
    ]
)

arbres_utilisés = model_conv.best_iteration_
print(f'Arbres utilisés : {arbres_utilisés:,} / {params_convergé["n_estimators"]:,}')
print(f'→ Early stopping {"déclenché" if arbres_utilisés < params_convergé["n_estimators"] else "NON déclenché — augmenter encore n_estimators"}')

# ── Évaluation ────────────────────────────────────────────────────────────────
pred_conv_trainval = model_conv.predict(X_trainval)
pred_conv_test     = model_conv.predict(X_test)

print('\n=== Modèle convergé — Résultats ===\n')
r2_conv_train, rmse_conv_train, _ = metriques(Y_trainval, pred_conv_trainval, 'Train+Val')
r2_conv_test,  rmse_conv_test,  _ = metriques(Y_test,     pred_conv_test,     'Test')
print(f'\nOverfit gap : {r2_conv_train - r2_conv_test:.4f}')

# ── Comparaison des 3 modèles ─────────────────────────────────────────────────
print('\n=== Comparaison Baseline → Optimisé → Convergé (Test) ===\n')
print(f'{"Modèle":20} {"R²":>8} {"RMSE (kWh)":>12} {"MAE (kWh)":>11}')
print('-' * 55)

for label, r2, rmse, mae in [
    ('Baseline',         r2_test,       rmse_test,       mae_test),
    ('Optimisé (3k)',    r2_opt_test,   rmse_opt_test,   None),
    ('Convergé (10k)',   r2_conv_test,  rmse_conv_test,  None),
]:
    mae_str = f'{mae:11.0f}' if mae else f'{"—":>11}'
    print(f'{label:20} {r2:8.4f} {rmse:12.0f} {mae_str}')

# Erreur relative (RMSE / mean)
mean_y = Y_test.mean()
print(f'\nMoyenne Y_test : {mean_y:.0f} kWh')
print(f'Erreur relative RMSE convergé : {rmse_conv_test / mean_y * 100:.1f}%')

### Réentraînement final avec les hyperparamètres K-Fold

In [ ]:
from sklearn.model_selection import KFold

# K-Fold : découper X_trainval en 5 parties égales
# À chaque essai Optuna, on entraîne 5 fois et on moyenne les RMSE
# → les hyperparamètres sont robustes à la partition, pas optimisés pour une seule val

def objective_kfold(trial):
    params = {
        'n_estimators'     : 3000,

        # Taux d'apprentissage plus bas → convergence plus fine, moins d'overfit
        'learning_rate'    : trial.suggest_float('learning_rate', 0.005, 0.05, log=True),

        # num_leaves réduit → arbres moins profonds → moins de mémorisation
        'num_leaves'       : trial.suggest_int('num_leaves', 20, 100),

        # min_child_samples élevé → chaque feuille doit représenter au moins N logements
        'min_child_samples': trial.suggest_int('min_child_samples', 50, 500),

        'colsample_bytree' : trial.suggest_float('colsample_bytree', 0.4, 0.9),
        'subsample'        : trial.suggest_float('subsample', 0.4, 0.9),

        # Régularisation bien plus forte qu'avant
        'reg_lambda'       : trial.suggest_float('reg_lambda', 1.0, 100.0, log=True),
        'reg_alpha'        : trial.suggest_float('reg_alpha',  0.1,  20.0, log=True),

        'random_state': 42, 'n_jobs': -1, 'verbose': -1,
    }

    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    rmse_folds = []

    for fold, (idx_tr, idx_vl) in enumerate(kf.split(X_trainval)):
        X_ktr = X_trainval.iloc[idx_tr];  Y_ktr = Y_trainval.iloc[idx_tr]
        X_kvl = X_trainval.iloc[idx_vl];  Y_kvl = Y_trainval.iloc[idx_vl]

        m = lgb.LGBMRegressor(**params)
        m.fit(X_ktr, Y_ktr,
              eval_set=[(X_kvl, Y_kvl)],
              callbacks=[lgb.early_stopping(50, verbose=False)])

        rmse_folds.append(np.sqrt(mean_squared_error(Y_kvl, m.predict(X_kvl))))

    return np.mean(rmse_folds)   # RMSE moyen sur 5 folds → plus stable


study_kf = optuna.create_study(direction='minimize')
study_kf.optimize(objective_kfold, n_trials=50, show_progress_bar=True)

print(f'\n=== Meilleurs hyperparamètres (K-Fold) ===')
print(f'RMSE CV moyen : {study_kf.best_value:.0f} kWh')
for k, v in study_kf.best_params.items():
    print(f'  {k:25} : {v}')

In [ ]:
# Réentraînement final avec les hyperparamètres K-Fold sur tout X_trainval
params_kf = dict(study_kf.best_params)
params_kf.update({
    'n_estimators' : 10_000,
    'random_state' : 42,
    'n_jobs'       : -1,
    'verbose'      : -1,
})

model_kf = lgb.LGBMRegressor(**params_kf)

model_kf.fit(
    X_trainval, Y_trainval,
    eval_set  = [(X_test, Y_test)],
    callbacks = [
        lgb.early_stopping(100, verbose=False),
        lgb.log_evaluation(500),
    ]
)

arbres_kf = model_kf.best_iteration_
print(f'Arbres utilisés : {arbres_kf:,} / {params_kf["n_estimators"]:,}')

pred_kf_trainval = model_kf.predict(X_trainval)
pred_kf_test     = model_kf.predict(X_test)

print('\n=== Modèle K-Fold — Résultats ===\n')
r2_kf_train, rmse_kf_train, _ = metriques(Y_trainval, pred_kf_trainval, 'Train+Val')
r2_kf_test,  rmse_kf_test,  _ = metriques(Y_test,     pred_kf_test,     'Test')
print(f'\nOverfit gap : {r2_kf_train - r2_kf_test:.4f}')

# ── Tableau récapitulatif final ───────────────────────────────────────────────
print('\n=== Progression — tous les modèles (Test) ===\n')
print(f'{"Modèle":25} {"R² test":>8} {"RMSE (kWh)":>12} {"Overfit gap":>13}')
print('-' * 62)
print(f'{"Baseline":25} {r2_test:8.4f} {rmse_test:12.0f}  {"—":>12}')
print(f'{"Optimisé Optuna (3k)":25} {r2_opt_test:8.4f} {rmse_opt_test:12.0f}  {"—":>12}')
print(f'{"Convergé (10k)":25} {r2_conv_test:8.4f} {rmse_conv_test:12.0f}  {r2_conv_train - r2_conv_test:12.4f}')
print(f'{"K-Fold + Régul forte":25} {r2_kf_test:8.4f} {rmse_kf_test:12.0f}  {r2_kf_train - r2_kf_test:12.4f}')

mean_y = Y_test.mean()
print(f'\nErreur relative RMSE K-Fold : {rmse_kf_test / mean_y * 100:.1f}%  (objectif < 10%)')